# Etapa 15 · Verificación cuantitativa

## Resumen
Motor sobre evidencia congelada: conversiones, QoQ/YoY y seis puentes contables. Cinco contrastes entre CASK y gasto/ASK permanecen sin reconciliar; no se explica con ellos el margen financiero.

## Contexto y método
Se trabaja exclusivamente con el paquete temporal aprobado de 2T26.

### Supuestos
1 milla = 1.609344 km. Redondeo conservador según la precisión conservada en Silver; se propagan intervalos. El costo efectivo no es precio de mercado. Una identidad matemática no demuestra causalidad.

In [1]:
from pathlib import Path
import os,sys,json,hashlib
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/analysis_agent/quantitative.py').exists())
os.chdir(root);sys.path.insert(0,str(root))
from src.analysis_agent.quantitative import build,OUTPUT
from src.config import PATHS
result=json.loads((OUTPUT/'calculations.json').read_bytes())
package_path=root/'analysis_runs/evidence'/(result['package_id']+'.json')
package=json.loads(package_path.read_bytes())

## Datos y reproducibilidad
Originales: versión inmutable en `analysis_runs/evidence`, documentos Bronze e IDs incorporados en el paquete.

In [2]:
def fingerprints():
    return {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in PATHS.gold.glob('*.parquet')}
before=fingerprints();original=package_path.read_bytes()
rebuilt=build(package)
assert rebuilt==result
assert package_path.read_bytes()==original
assert fingerprints()==before
print('Cálculos idénticos; evidencia y Gold intactos.')

Cálculos idénticos; evidencia y Gold intactos.


## Resultados
Se distingue cierre aritmético de equivalencia entre definiciones.

In [3]:
from collections import Counter
print(Counter(c['status'] for c in result['checks']))
assert all(b['status']=='passed' for b in result['bridges']+result['identity_bridges'])
assert len([c for c in result['checks'] if c['status']=='not_reconciled'])==5
assert result['analysis_constraints']['allow_spread_to_financial_margin_attribution'] is False
print('Seis puentes contables reconciliados; cinco contrastes de alcance sin reconciliar.')

Counter({'passed': 35, 'not_reconciled': 5})
Seis puentes contables reconciliados; cinco contrastes de alcance sin reconciliar.


## Conclusiones
Los resultados están listos para revisión humana. Las diferencias de costo/ASK no se forzaron ni se trataron como errores de redondeo. El analista deberá usar las restricciones del motor y conservar las referencias por cifra. No hay redacción ni publicación autorizada por esta ejecución.